In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

In [2]:
import sys
sys.path.append("/kaggle/input/hull-tactical-market-prediction")

import kaggle_evaluation.default_inference_server

In [3]:
# ============================================================
# Kaggle Submission Cell (XGB + ExtraTrees + LGB + XGB2) + AUDIT DEBUG
#   - RAW ensemble: blend raw preds -> Kelly
#   - Kelly std from MOST RECENT 70% of y_train (tail)
#   - XGB + XGB2: early stopping + (train+eval) sample_weight + predict(best_iteration)
#   - LGB: early stopping callback + predict(best_iteration_)  [NO sample_weight EVER]
#   - Extra: supports sample_weight
#   - Robust streaming history:
#       * Local gateway overlap (test date_id within train range): seed ONLY from past (< first test date_id)
#       * Real rerun (future date_id > max(train)): seed from end of train
#       * Always compute features on (history + current) then FILTER back to current date_id(s)
# ============================================================

import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, cast

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor

import kaggle_evaluation.default_inference_server

# =====================
# Debug controls
# =====================
DEBUG = True
DEBUG_PRED_CALLS = 8
_pred_calls = 0

def dprint(*args, **kwargs):
    if DEBUG:
        print(*args, **kwargs)

# =====================
# Paths and constants
# =====================
DATA_PATH = Path("/kaggle/input/hull-tactical-market-prediction")
MODEL_NAME = "xgb_extra_lgb_xgb2_raw_ensemble"

# Feature groups used by the models in the training script
XGB_GROUPS: Sequence[str] = [
    "lags",
    "squares",
    "roll_min",
    "roll_max",
    "ema",
    "rel",
    "prod",
    "div",
    "zscore",
    "roll_mean",
    "cubes",
]

# Explicit, ordered feature lists used by the models
MODEL_EXPLICIT_FEATURE_LIST: Dict[str, List[str]] = {
    "xgb": [
        "M4_lag_5",
        "V9_sq",
        "M4_roll_max_3",
        "V9_div_roll_mean_2",
        "M4_roll_min_2",
        "M4_lag_1",
        "V9_prod_E3",
        "P10_minus_roll_mean_5",
        "V13_div_roll_mean_2",
        "M7_minus_roll_mean_60",
        "M4_ema_3",
        "M4_lag_2",
        "M1_minus_roll_mean_2",
        "V9_lag_3",
        "M7_div_roll_mean_60",
        "M4_roll_max_2",
        "P10_div_roll_mean_5",
        "S2_minus_roll_mean_3",
    ],
    "extra": [
        "V9_prod_E3",
        "V9_lag_3",
        "V9_lag_1",
        "V9_ema_3",
        "V9_div_roll_mean_2",
        "V9_div_M9",
        "V7_lag_1",
        "V7_div_P4",
        "V13_prod_M4",
        "V13_prod_E19",
        "V13_minus_roll_mean_2",
        "V13_lag_3",
        "V13_lag_2",
        "V13_div_roll_mean_3",
        "V13_div_P4",
        "V13_div_M9",
        "S5_div_P8",
        "S5_div_P7",
        "S2_minus_roll_mean_3",
        "S2_div_roll_mean_3",
        "P8_div_P13",
        "P6_roll_z_2",
        "P5_lag_3",
        "P5_lag_1",
        "P4_lag_5",
        "P2_roll_z_10",
        "P11_minus_roll_mean_5",
        "P10_minus_roll_mean_5",
        "P10_div_roll_mean_5",
        "M7_div_roll_mean_60",
        "M4_lag_2",
        "M4_ema_20",
        "M4_ema_10",
        "M4_div_roll_mean_3",
        "M4_div_roll_mean_2",
        "M4_div_P6",
        "M4_div_I5",
        "M3_div_P10",
        "M2_ema_5",
        "M1_roll_z_3",
        "M1_minus_roll_mean_2",
        "M1_div_roll_mean_2",
        "M1_div_P8",
        "M12_lag_90",
        "M12_lag_10",
        "I4_lag_20",
        "I2_roll_z_3",
        "I2_roll_z_2",
        "I2_div_E16",
        "E3_div_V10",
        "E19_roll_z_3",
        "E19_lag_90",
        "E19_lag_5",
        "E19_lag_2",
        "E19_lag_1",
        "E19_ema_5",
        "E19_ema_3",
        "E19_ema_2",
    ],
    "lgb": [
        "P10_roll_min_5",
        "P10_roll_min_10",
        "P10_sq",
        "I2_sq",
        "P10_roll_mean_5",
        "E19_roll_max_2",
        "E19_roll_max_3",
        "E19_ema_2",
        "M4_ema_10",
        "E19_div_roll_mean_2",
        "E19_div_roll_mean_3",
        "E19_minus_roll_mean_3",
        "M1_div_roll_mean_2",
        "V13_minus_roll_mean_2",
        "V13_minus_roll_mean_3",
        "V9_div_roll_mean_2",
        "E19_prod_P10",
        "V13_prod_E19",
        "V13_prod_M4",
    ],
}

# Sample-weight configuration matching sample_weighting.py semantics
BAD_START = 7500
BAD_END = 8300
RECENCY_BOOST = True
BAD_SCALE = 0.1

# =============================================================================
# Feature engineering helpers
# =============================================================================

LAG_SPECS: Dict[str, list[int]] = {
    "V13": [1, 2, 3],
    "M4":  [1, 2, 3, 5],
    "V7":  [1, 2],
    "E19": [1, 2, 5, 90],
    "P12": [20],
    "I4":  [20],
    "M5":  [180],
    "P10": [2],
    "M12": [90, 10],
    "V9":  [1, 3],
    "M3":  [2],
    "P5":  [1, 3],
    "P4":  [5],
    "S8":  [1],
}

BASE_TO_SQ: Dict[str, str] = {
    "V9":  "V9_sq",
    "M4":  "M4_sq",
    "E19": "E19_sq",
    "P10": "P10_sq",
    "P7":  "P7_sq",
    "V13": "V13_sq",
    "V7":  "V7_sq",
    "E2":  "E2_sq",
    "S5":  "S5_sq",
    "M3":  "M3_sq",
    "P6":  "P6_sq",
    "P2":  "P2_sq",
    "I2":  "I2_sq",
    "E16": "E16_sq",
    "P12": "P12_sq",
    "M9":  "M9_sq",
    "P11": "P11_sq",
}

BASE_TO_CUB: Dict[str, str] = {
    "P10": "P10_cub",
    "V9":  "V9_cub",
    "V7":  "V7_cub",
    "P11": "P11_cub",
    "E3":  "E3_cub",
    "I2":  "I2_cub",
    "S8":  "S8_cub",
    "V10": "V10_cub",
    "P2":  "P2_cub",
    "V13": "V13_cub",
    "E19": "E19_cub",
    "M4":  "M4_cub",
    "P8":  "P8_cub",
    "E17": "E17_cub",
    "E18": "E18_cub",
    "E16": "E16_cub",
    "E2":  "E2_cub",
}

ROLLING_MEAN_SPECS: Dict[str, list[int]] = {
    "V13": [2, 3],
    "M4":  [2, 3],
    "E19": [2, 3],
    "S2":  [3],
    "P10": [5],
    "P11": [5],
    "M11": [180],
    "V9":  [2],
    "I4":  [20],
    "M7":  [60],
    "M1":  [2],
}

REL_MINUS_SPECS: Dict[str, list[int]] = dict(ROLLING_MEAN_SPECS)
REL_DIV_SPECS: Dict[str, list[int]] = dict(ROLLING_MEAN_SPECS)

ROLLING_STD_SPECS: Dict[str, list[int]] = {
    "E19": [3],
    "I2":  [2, 3],
    "P6":  [2],
    "M4":  [2],
    "S5":  [2],
    "S10": [2],
    "P2":  [10],
    "M1":  [3],
}

EMA_SPECS: Dict[str, list[int]] = {
    "M4":  [2, 3, 5, 10, 20],
    "E19": [2, 3, 5],
    "V13": [3, 5],
    "M3":  [2, 10],
    "V9":  [2, 3, 10],
    "P7":  [10, 180],
    "M2":  [5, 10],
}

DIV_PAIRS: List[Tuple[str, str]] = [
    ("E19", "P4"),
    ("P6",  "M12"),
    ("V13", "P4"),
    ("P8",  "P13"),
    ("V7",  "P4"),
    ("S5",  "P8"),
    ("I2",  "E16"),
    ("M3",  "P10"),
    ("V13", "M9"),
    ("P2",  "E17"),
    ("M4",  "I5"),
    ("E19", "P3"),
    ("V13", "S9"),
    ("M1",  "P8"),
    ("V9",  "M9"),
    ("E3",  "V10"),
    ("M9",  "V10"),
    ("M4",  "P6"),
    ("S5",  "P7"),
]

MINUS_PAIRS: List[Tuple[str, str]] = [
    ("P8",  "M2"),
    ("E19", "P6"),
    ("V13", "E19"),
    ("V7",  "I2"),
    ("V13", "S2"),
    ("V9",  "E3"),
    ("V13", "M1"),
    ("S9",  "E2"),
    ("E19", "I2"),
    ("S12", "P2"),
    ("M4",  "M3"),
    ("V7",  "M3"),
    ("V7",  "P2"),
    ("V9",  "M1"),
    ("V7",  "S2"),
    ("V9",  "P2"),
    ("P5",  "P13"),
    ("E19", "P7"),
    ("M4",  "E19"),
    ("S8",  "M2"),
]

ROLLING_MIN_SPECS: Dict[str, list[int]] = {
    "E19": [2, 3],
    "M4":  [2, 3],
    "V13": [3],
    "P10": [5, 10],
}

ROLLING_MAX_SPECS: Dict[str, list[int]] = {
    "E19": [2, 3],
    "M4":  [2, 3],
    "V13": [3],
    "P10": [5],
}

PRODUCT_PAIRS: List[Tuple[str, str]] = [
    ("V13", "E19"),
    ("V13", "M4"),
    ("V9",  "E3"),
    ("V7",  "P2"),
    ("E19", "P10"),
]

# ---- Polars helpers ----
def _rolling_mean(expr: pl.Expr, window: int, min_samples: int = 1) -> pl.Expr:
    e = cast(Any, expr)
    return e.rolling_mean(window_size=window, min_samples=min_samples)

def _rolling_std(expr: pl.Expr, window: int, min_samples: int = 1) -> pl.Expr:
    e = cast(Any, expr)
    return e.rolling_std(window_size=window, min_samples=min_samples)

def _rolling_min(expr: pl.Expr, window: int, min_samples: int = 1) -> pl.Expr:
    e = cast(Any, expr)
    return e.rolling_min(window_size=window, min_samples=min_samples)

def _rolling_max(expr: pl.Expr, window: int, min_samples: int = 1) -> pl.Expr:
    e = cast(Any, expr)
    return e.rolling_max(window_size=window, min_samples=min_samples)

def _ewm_mean(expr: pl.Expr, span: int, adjust: bool = False, min_samples: int = 1) -> pl.Expr:
    e = cast(Any, expr)
    return e.ewm_mean(span=span, adjust=adjust, min_samples=min_samples)

def add_engineered_features(df: pl.DataFrame, groups: Optional[Sequence[str]]) -> pl.DataFrame:
    if "date_id" not in df.columns:
        return df

    df = df.sort("date_id")
    gset = set(groups) if groups is not None else None

    def use(group_key: str) -> bool:
        return gset is None or group_key in gset

    if use("day") and "date_id" in df.columns:
        df = df.with_columns((pl.col("date_id") % 5).cast(pl.Float64).alias("day_of_cycle"))

    if use("lags"):
        for base_col, lag_list in LAG_SPECS.items():
            if base_col in df.columns:
                for lag in sorted(set(lag_list)):
                    df = df.with_columns(
                        pl.col(base_col).cast(pl.Float64, strict=False).shift(lag).alias(f"{base_col}_lag_{lag}")
                    )

    if use("squares"):
        for base_col, new_name in BASE_TO_SQ.items():
            if base_col in df.columns:
                df = df.with_columns((pl.col(base_col).cast(pl.Float64, strict=False) ** 2).alias(new_name))

    if use("cubes"):
        for base_col, new_name in BASE_TO_CUB.items():
            if base_col in df.columns:
                df = df.with_columns((pl.col(base_col).cast(pl.Float64, strict=False) ** 3).alias(new_name))

    if use("roll_mean"):
        for base_col, windows in ROLLING_MEAN_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for window in sorted(set(windows)):
                    df = df.with_columns(_rolling_mean(x, window).alias(f"{base_col}_roll_mean_{window}"))

    if use("roll_std"):
        for base_col, windows in ROLLING_STD_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for window in sorted(set(windows)):
                    df = df.with_columns(_rolling_std(x, window).alias(f"{base_col}_roll_std_{window}"))

    if use("roll_min"):
        for base_col, windows in ROLLING_MIN_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for window in sorted(set(windows)):
                    df = df.with_columns(_rolling_min(x, window).alias(f"{base_col}_roll_min_{window}"))

    if use("roll_max"):
        for base_col, windows in ROLLING_MAX_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for window in sorted(set(windows)):
                    df = df.with_columns(_rolling_max(x, window).alias(f"{base_col}_roll_max_{window}"))

    if use("ema"):
        for base_col, spans in EMA_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for span in sorted(set(spans)):
                    df = df.with_columns(_ewm_mean(x, span).alias(f"{base_col}_ema_{span}"))

    if use("div"):
        eps = 1e-6
        for col1, col2 in DIV_PAIRS:
            if col1 in df.columns and col2 in df.columns:
                num_expr = pl.col(col1).cast(pl.Float64, strict=False)
                denom_expr = pl.col(col2).cast(pl.Float64, strict=False)
                safe_denom = pl.when(denom_expr == 0).then(eps).otherwise(denom_expr) + eps
                df = df.with_columns((num_expr / safe_denom).alias(f"{col1}_div_{col2}"))

    if use("minus"):
        for col1, col2 in MINUS_PAIRS:
            if col1 in df.columns and col2 in df.columns:
                df = df.with_columns(
                    (pl.col(col1).cast(pl.Float64, strict=False) - pl.col(col2).cast(pl.Float64, strict=False)).alias(f"{col1}_minus_{col2}")
                )

    if use("zscore"):
        eps = 1e-6
        for base_col, windows in ROLLING_STD_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for window in sorted(set(windows)):
                    roll_mean = _rolling_mean(x, window)
                    roll_std = _rolling_std(x, window)
                    df = df.with_columns(((x - roll_mean) / (roll_std + eps)).alias(f"{base_col}_roll_z_{window}"))

    if use("rel"):
        eps = 1e-6
        for base_col, windows in REL_MINUS_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for window in sorted(set(windows)):
                    roll_mean = _rolling_mean(x, window)
                    df = df.with_columns((x - roll_mean).alias(f"{base_col}_minus_roll_mean_{window}"))

        for base_col, windows in REL_DIV_SPECS.items():
            if base_col in df.columns:
                x = pl.col(base_col).cast(pl.Float64, strict=False)
                for window in sorted(set(windows)):
                    roll_mean = _rolling_mean(x, window)
                    df = df.with_columns((x / (roll_mean + eps)).alias(f"{base_col}_div_roll_mean_{window}"))

    if use("prod"):
        for col1, col2 in PRODUCT_PAIRS:
            if col1 in df.columns and col2 in df.columns:
                df = df.with_columns(
                    (pl.col(col1).cast(pl.Float64, strict=False) * pl.col(col2).cast(pl.Float64, strict=False)).alias(f"{col1}_prod_{col2}")
                )

    return df

TARGET_COLS = {"market_forward_excess_returns", "forward_returns", "risk_free_rate"}
NON_FEATURE_COLS = {"is_scored", "date_id"} | TARGET_COLS

def build_base_features(data_path: Path, eng_groups: Sequence[str], include_raw_base: bool = True):
    train_raw = pl.read_csv(data_path / "train.csv")
    test_raw = pl.read_csv(data_path / "test.csv")

    raw_common_cols = set(train_raw.columns) & set(test_raw.columns)
    base_candidate_cols = sorted(raw_common_cols - NON_FEATURE_COLS)

    train_eng = add_engineered_features(train_raw, eng_groups)
    test_eng = add_engineered_features(test_raw, eng_groups)

    train_eng = train_eng.with_columns(pl.all().exclude("date_id").cast(pl.Float64, strict=False))

    train_filled = train_eng.with_columns([
        pl.when(pl.col(c).is_null()).then(pl.col(c).mean()).otherwise(pl.col(c)).alias(c)
        for c in train_eng.columns if c != "date_id"
    ])

    train_cols = set(train_filled.columns)
    test_cols = set(test_eng.columns)
    common_cols = train_cols & test_cols
    all_candidate_feats = common_cols - NON_FEATURE_COLS

    base_feature_cols_set = set(base_candidate_cols) & all_candidate_feats
    engineered_feature_cols_set = all_candidate_feats - base_feature_cols_set

    feature_cols_set = engineered_feature_cols_set if not include_raw_base else (base_feature_cols_set | engineered_feature_cols_set)

    feature_cols = sorted(feature_cols_set)
    train_sorted = train_filled.sort("date_id")

    y = train_sorted["market_forward_excess_returns"].to_numpy()
    forward_returns_all = train_sorted["forward_returns"].to_numpy()
    risk_free_all = train_sorted["risk_free_rate"].to_numpy()

    X = train_sorted.select(feature_cols).to_pandas()
    X = X.fillna(X.mean())

    test_engineered = test_eng.select(feature_cols).to_pandas()
    return X, y, forward_returns_all, risk_free_all, feature_cols, test_engineered

def apply_feature_override(model_name: str, X: pd.DataFrame) -> pd.DataFrame:
    required = MODEL_EXPLICIT_FEATURE_LIST[model_name]
    missing = [c for c in required if c not in X.columns]
    if missing:
        raise ValueError(f"[feature_override:{model_name}] Missing expected columns: {missing}")
    return X.loc[:, required]

def build_sample_weights(
    n_samples: int,
    bad_start: int = BAD_START,
    bad_end: int = BAD_END,
    recency_boost: bool = RECENCY_BOOST,
    bad_scale: float = BAD_SCALE,
) -> np.ndarray:
    date_id = np.arange(n_samples) + 1
    bad_mask = (date_id >= bad_start) & (date_id <= bad_end)
    w_quality = np.where(bad_mask, bad_scale, 1.0)

    if recency_boost and n_samples > 1:
        ranks = np.arange(n_samples, dtype=float)
        w_age = 1.0 + ranks / ranks.max()
    else:
        w_age = np.ones(n_samples, dtype=float)

    return w_quality * w_age

KELLY_RET_STD: float | None = None

def set_kelly_ret_std(value: float) -> None:
    global KELLY_RET_STD
    KELLY_RET_STD = float(value)

def alloc_kelly(pred: np.ndarray) -> np.ndarray:
    if KELLY_RET_STD is None:
        raise RuntimeError("KELLY_RET_STD is not set.")
    sigma2 = KELLY_RET_STD ** 2 + 1e-12
    f_raw = (np.asarray(pred, dtype=float)) / sigma2
    f_clipped = np.clip(f_raw, -1.0, 1.0)
    a = 1.0 + f_clipped
    return np.clip(a, 0.0, 2.0)

def _predict_xgb_best_iter(model: XGBRegressor, X):
    best_it = getattr(model, "best_iteration", None)
    if best_it is not None:
        return model.predict(X, iteration_range=(0, int(best_it) + 1))
    return model.predict(X)

def _predict_lgb_best_iter(model: LGBMRegressor, X):
    best_it = getattr(model, "best_iteration_", None)
    if best_it is not None and int(best_it) > 0:
        return model.predict(X, num_iteration=int(best_it))
    return model.predict(X)

# =============================================================================
# Train the models and pre-compute test features
# =============================================================================

X_base_full, y_full, fwd_full, rf_full, feature_cols, test_features_raw = build_base_features(
    DATA_PATH,
    eng_groups=XGB_GROUPS,
    include_raw_base=True,
)

# ---- ONLY TRAIN ON MOST RECENT 70% OF TOTAL TRAIN ----
TAIL_FRAC = 0.70
N_full = len(y_full)
tail_start = int(N_full * (1.0 - TAIL_FRAC))

X_base  = X_base_full.iloc[tail_start:].reset_index(drop=True)
y_train = y_full[tail_start:]
fwd_train = fwd_full[tail_start:]
rf_train  = rf_full[tail_start:]

sample_weight_full = build_sample_weights(n_samples=N_full)
sample_weight = sample_weight_full[tail_start:]

feature_means = X_base.mean()

X_train_xgb   = apply_feature_override("xgb", X_base).fillna(feature_means)
X_train_xgb2  = X_train_xgb
X_train_extra = apply_feature_override("extra", X_base).fillna(feature_means)
X_train_lgb   = apply_feature_override("lgb", X_base).fillna(feature_means)

XGB_BEST = dict(
    learning_rate=0.04,
    max_depth=5,
    min_child_weight=1.0,
    subsample=1.0,
    colsample_bytree=0.7,
    reg_lambda=3.0,
    reg_alpha=0.0,
)

XGB2_BEST = dict(
    learning_rate=0.05555508154466528,
    max_depth=10,
    min_child_weight=8.4154264341855,
    subsample=0.9104958025718234,
    colsample_bytree=0.9796819880055834,
    reg_lambda=0.004278523789494383,
    reg_alpha=0.0003107263331093944,
    gamma=4.3175317797788444e-05,
    max_delta_step=3,
)

LGB_BEST = dict(
    learning_rate=0.005,
    num_leaves=127,
    min_child_samples=25,
    subsample=1.0,
    colsample_bytree=0.85,
    reg_lambda=8.0,
    reg_alpha=0.0,
    min_split_gain=0.0,
)

N_ESTIMATORS = 12000
EARLY_STOPPING = 200

# RAW ensemble weights (blend raw preds -> Kelly)
W_RAW_XGB, W_RAW_LGB, W_RAW_EXTRA, W_RAW_XGB2 = 0.08, 0.10, 0.52, 0.30

# Kelly std from MOST RECENT 70% (y_train is already the tail)
set_kelly_ret_std(float(np.std(y_train)))

# =====================
# AUDIT prints (train)
# =====================
dprint("=== PATH CHECK ===")
dprint("DATA_PATH =", DATA_PATH)
dprint("MODEL_NAME =", MODEL_NAME)

dprint("=== TRAIN BUILD ===")
dprint("X_base_full:", X_base_full.shape, "y_full:", y_full.shape)
dprint("TAIL_FRAC:", TAIL_FRAC, "N_full:", N_full, "tail_start:", tail_start)
dprint("X_base:", X_base.shape, "y_train:", y_train.shape)

dprint("=== FEATURE COLS ===")
dprint("feature_cols (all candidates):", len(feature_cols))
for k in ["xgb", "extra", "lgb"]:
    dprint(f"{k} explicit feats:", len(MODEL_EXPLICIT_FEATURE_LIST[k]))

dprint("=== WEIGHTS ===")
dprint("W_RAW:", (W_RAW_XGB, W_RAW_LGB, W_RAW_EXTRA, W_RAW_XGB2),
       "sum=", (W_RAW_XGB + W_RAW_LGB + W_RAW_EXTRA + W_RAW_XGB2))

dprint("=== SAMPLE WEIGHTS ===")
dprint("sample_weight:", "min", float(sample_weight.min()), "max", float(sample_weight.max()),
       "mean", float(sample_weight.mean()))
dprint("bad window:", (BAD_START, BAD_END), "bad_scale:", BAD_SCALE, "recency_boost:", RECENCY_BOOST)

dprint("=== KELLY ===")
dprint("KELLY_RET_STD from y_train tail:", float(np.std(y_train)))

# ---- XGB (weighted) ----
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    n_estimators=N_ESTIMATORS,
    **XGB_BEST,
)

n = len(y_train)
eval_n = max(1, int(0.10 * n))
tr_n = n - eval_n
_tr_idx = np.arange(tr_n)
_va_idx = np.arange(tr_n, n)

X_tr_xgb = X_train_xgb.iloc[_tr_idx]
y_tr = y_train[_tr_idx]
w_tr = sample_weight[_tr_idx]

X_va_xgb = X_train_xgb.iloc[_va_idx]
y_va = y_train[_va_idx]
w_va = sample_weight[_va_idx]

xgb_model.fit(
    X_tr_xgb,
    y_tr,
    eval_set=[(X_va_xgb, y_va)],
    early_stopping_rounds=EARLY_STOPPING,
    verbose=False,
    sample_weight=w_tr,
    sample_weight_eval_set=[w_va],
)

# ---- XGB2 (weighted; same split/weights as XGB) ----
xgb2_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    n_estimators=N_ESTIMATORS,
    **XGB2_BEST,
)

X_tr_xgb2 = X_train_xgb2.iloc[_tr_idx]
X_va_xgb2 = X_train_xgb2.iloc[_va_idx]

xgb2_model.fit(
    X_tr_xgb2,
    y_tr,
    eval_set=[(X_va_xgb2, y_va)],
    early_stopping_rounds=EARLY_STOPPING,
    verbose=False,
    sample_weight=w_tr,
    sample_weight_eval_set=[w_va],
)

# ---- Extra (weighted) ----
extra_model = ExtraTreesRegressor(
    n_estimators=1200,
    max_depth=30,
    min_samples_leaf=3,
    min_samples_split=10,
    max_features=1.0,
    bootstrap=True,
    max_samples=0.7,
    n_jobs=-1,
    random_state=42,
)
extra_model.fit(X_train_extra, y_train, sample_weight=sample_weight)

# ---- LGB (NO sample_weight EVER) ----
lgb_model = LGBMRegressor(
    objective="regression",
    metric="rmse",
    boosting_type="gbdt",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    n_estimators=N_ESTIMATORS,
    **LGB_BEST,
)

X_tr_lgb = X_train_lgb.iloc[_tr_idx]
X_va_lgb = X_train_lgb.iloc[_va_idx]

lgb_model.fit(
    X_tr_lgb,
    y_tr,
    eval_set=[(X_va_lgb, y_va)],
    callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)],
)

dprint("=== FIT RESULTS ===")
dprint("XGB best_iteration:", getattr(xgb_model, "best_iteration", None))
dprint("XGB2 best_iteration:", getattr(xgb2_model, "best_iteration", None))
dprint("LGB best_iteration_:", getattr(lgb_model, "best_iteration_", None))

# sanity check: offline test rows should match test_date_ids length
test_date_ids = pd.read_csv(DATA_PATH / "test.csv", usecols=["date_id"]).squeeze("columns")
if len(test_date_ids) != len(test_features_raw):
    raise ValueError("test date_id length does not match engineered test feature rows")

dprint("=== TEST SANITY ===")
dprint("test_date_ids:", len(test_date_ids), "test_features_raw:", len(test_features_raw))

# =============================================================================
# Robust streaming history buffer (works for local overlap + real future)
# =============================================================================

HIST_SEED_N = 400
HIST_MAX_ROWS = 1200

HIST_RAW: Optional[pl.DataFrame] = None
STREAM_COLS: List[str] = []
STREAM_SCHEMA: Dict[str, pl.DataType] = {}
TRAIN_MAX_DATE_ID: int = -1
TRAIN_STREAM: Optional[pl.DataFrame] = None

def _init_stream_from_train() -> None:
    """Initialize STREAM_COLS/schema and keep a sorted train slice for seeding history."""
    global STREAM_COLS, STREAM_SCHEMA, TRAIN_MAX_DATE_ID, TRAIN_STREAM
    train_raw = pl.read_csv(DATA_PATH / "train.csv")
    # stream cols = date_id + all non-target cols except is_scored
    cols = [c for c in train_raw.columns if c not in TARGET_COLS and c != "is_scored"]
    if "date_id" not in cols:
        cols = ["date_id"] + cols
    # keep order stable: date_id first, then others as in file
    cols = ["date_id"] + [c for c in cols if c != "date_id"]

    train_stream = train_raw.select(cols).sort("date_id")

    STREAM_COLS = cols
    STREAM_SCHEMA = {c: train_stream.schema[c] for c in STREAM_COLS}
    TRAIN_MAX_DATE_ID = int(train_stream.select(pl.max("date_id")).item())
    TRAIN_STREAM = train_stream

def _coerce_stream_schema(df: pl.DataFrame) -> pl.DataFrame:
    """Coerce df to exactly STREAM_COLS with train-aligned dtypes (prevents concat dtype errors)."""
    # add missing cols as nulls
    for c in STREAM_COLS:
        if c not in df.columns:
            df = df.with_columns(pl.lit(None, dtype=STREAM_SCHEMA[c]).alias(c))
    df = df.select(STREAM_COLS)

    # cast to expected dtypes (strict=False so bad parses become null)
    cast_exprs = []
    for c in STREAM_COLS:
        dt = STREAM_SCHEMA[c]
        if c == "date_id":
            cast_exprs.append(pl.col(c).cast(pl.Int64, strict=False).alias(c))
        else:
            cast_exprs.append(pl.col(c).cast(dt, strict=False).alias(c))
    return df.with_columns(cast_exprs)

def _seed_history_for_first_call(cur_min_date_id: int) -> pl.DataFrame:
    """
    If local gateway overlap (cur_min <= TRAIN_MAX_DATE_ID): seed ONLY from past (< cur_min).
    If real rerun (cur_min > TRAIN_MAX_DATE_ID): seed from end of train.
    """
    if TRAIN_STREAM is None:
        _init_stream_from_train()
    assert TRAIN_STREAM is not None

    if cur_min_date_id <= TRAIN_MAX_DATE_ID:
        seed = TRAIN_STREAM.filter(pl.col("date_id") < cur_min_date_id).tail(HIST_SEED_N)
    else:
        seed = TRAIN_STREAM.tail(HIST_SEED_N)

    return _coerce_stream_schema(seed)

def _append_to_history(test_df: pl.DataFrame) -> pl.DataFrame:
    """Append current batch to history, keep sorted, unique date_id (last wins), and cap size."""
    global HIST_RAW
    if TRAIN_STREAM is None:
        _init_stream_from_train()

    cur_ids = test_df["date_id"].to_list() if "date_id" in test_df.columns else []
    cur_min = int(min(cur_ids)) if cur_ids else (TRAIN_MAX_DATE_ID + 1)

    if HIST_RAW is None:
        HIST_RAW = _seed_history_for_first_call(cur_min)

    # select + coerce current batch
    cur = _coerce_stream_schema(test_df)

    # concat + sort + unique by date_id (keep last) + cap
    combined = pl.concat([HIST_RAW, cur], how="vertical").sort("date_id")
    # if duplicates (possible in local gateway overlap), keep last occurrence
    combined = combined.unique(subset=["date_id"], keep="last").sort("date_id")
    if combined.height > HIST_MAX_ROWS:
        combined = combined.tail(HIST_MAX_ROWS)

    HIST_RAW = combined
    return HIST_RAW

# eager init so we can print stream info once
_init_stream_from_train()

dprint("=== STREAM BUFFER SEED ===")
dprint("STREAM_COLS:", len(STREAM_COLS))
dprint("HIST_SEED_N:", HIST_SEED_N, "HIST_MAX_ROWS:", HIST_MAX_ROWS)

# =============================================================================
# predict() for Kaggle server
# =============================================================================
def predict(test: pl.DataFrame):
    global _pred_calls
    _pred_calls += 1

    # --- debug header ---
    if DEBUG and _pred_calls <= DEBUG_PRED_CALLS:
        try:
            dprint(f"\n=== PRED CALL {_pred_calls} ===")
            dprint("incoming test shape:", test.shape)
            if "date_id" in test.columns:
                vals = test["date_id"].to_list()
                dprint("incoming date_id:", vals[:5], ("..." if len(vals) > 5 else ""))
            dprint("cols:", len(test.columns))
        except Exception as e:
            dprint("predict debug header error:", repr(e))

    # --- history append (robust for local overlap + real future) ---
    hist = _append_to_history(test)

    if DEBUG and _pred_calls <= DEBUG_PRED_CALLS:
        try:
            dprint("hist shape:", hist.shape)
            tail_ids = hist.select("date_id").tail(5)["date_id"].to_list()
            dprint("hist tail date_id:", tail_ids)
        except Exception as e:
            dprint("hist debug error:", repr(e))

    # --- engineer on full hist, then FILTER back to current row(s) ---
    cur_ids = test["date_id"].to_list() if "date_id" in test.columns else []
    cur_ids_set = set(int(x) for x in cur_ids)

    hist_eng = add_engineered_features(hist, XGB_GROUPS)
    cur_rows = hist_eng.filter(pl.col("date_id").is_in(list(cur_ids_set))).sort("date_id")

    if cur_rows.height == 0:
        # fallback: if something weird happened, use engineered on test-only (may have null lags)
        if DEBUG and _pred_calls <= DEBUG_PRED_CALLS:
            dprint("WARNING: cur_rows empty after filtering; falling back to test-only engineered rows.")
        cur_rows = add_engineered_features(_coerce_stream_schema(test), XGB_GROUPS).sort("date_id")

    test_base = cur_rows.select(feature_cols).to_pandas()

    if DEBUG and _pred_calls <= DEBUG_PRED_CALLS:
        dprint("engineered test_base shape:", test_base.shape)
        try:
            null_frac = test_base.isna().mean().sort_values(ascending=False)
            dprint("top null-frac cols (first 10):")
            dprint(null_frac.head(10))
        except Exception as e:
            dprint("null-frac debug error:", repr(e))

    # --- model feature overrides ---
    test_xgb   = apply_feature_override("xgb", test_base).fillna(feature_means)
    test_xgb2  = test_xgb
    test_extra = apply_feature_override("extra", test_base).fillna(feature_means)
    test_lgb   = apply_feature_override("lgb", test_base).fillna(feature_means)

    if DEBUG and _pred_calls <= DEBUG_PRED_CALLS:
        for name, df_ in [("xgb", test_xgb), ("xgb2", test_xgb2), ("extra", test_extra), ("lgb", test_lgb)]:
            try:
                frac = float(np.isnan(df_.to_numpy()).mean())
                dprint(f"{name}: post-fill NaN frac =", frac)
            except Exception as e:
                dprint(f"{name}: NaN-frac error:", repr(e))

    # --- raw preds ---
    raw_xgb   = _predict_xgb_best_iter(xgb_model, test_xgb)
    raw_xgb2  = _predict_xgb_best_iter(xgb2_model, test_xgb2)
    raw_extra = extra_model.predict(test_extra)
    raw_lgb   = _predict_lgb_best_iter(lgb_model, test_lgb)

    raw_pred_ens = (
        W_RAW_XGB   * np.asarray(raw_xgb, dtype=float) +
        W_RAW_LGB   * np.asarray(raw_lgb, dtype=float) +
        W_RAW_EXTRA * np.asarray(raw_extra, dtype=float) +
        W_RAW_XGB2  * np.asarray(raw_xgb2, dtype=float)
    )

    positions = alloc_kelly(raw_pred_ens)

    if DEBUG and _pred_calls <= DEBUG_PRED_CALLS:
        try:
            dprint("raw pred stats:",
                   "xgb", (float(np.min(raw_xgb)), float(np.max(raw_xgb))),
                   "xgb2", (float(np.min(raw_xgb2)), float(np.max(raw_xgb2))),
                   "extra", (float(np.min(raw_extra)), float(np.max(raw_extra))),
                   "lgb", (float(np.min(raw_lgb)), float(np.max(raw_lgb))))
            dprint("ens raw_pred_ens stats:", float(np.min(raw_pred_ens)), float(np.max(raw_pred_ens)))
            dprint("positions stats:", float(np.min(positions)), float(np.max(positions)))
        except Exception as e:
            dprint("pred stats debug error:", repr(e))

    # Kaggle expects float for single-row, array for multi-row
    if len(positions) == 1:
        return float(positions[0])
    return np.array(positions)

# =============================================================================
# Launch server
# =============================================================================
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    inference_server.run_local_gateway((str(DATA_PATH),))

=== PATH CHECK ===
DATA_PATH = /kaggle/input/hull-tactical-market-prediction
MODEL_NAME = xgb_extra_lgb_xgb2_raw_ensemble
=== TRAIN BUILD ===
X_base_full: (9048, 261) y_full: (9048,)
TAIL_FRAC: 0.7 N_full: 9048 tail_start: 2714
X_base: (6334, 261) y_train: (6334,)
=== FEATURE COLS ===
feature_cols (all candidates): 261
xgb explicit feats: 18
extra explicit feats: 58
lgb explicit feats: 19
=== WEIGHTS ===
W_RAW: (0.08, 0.1, 0.52, 0.3) sum= 1.0
=== SAMPLE WEIGHTS ===
sample_weight: min 0.1828893555874876 max 2.0 mean 1.4368080328777384
bad window: (7500, 8300) bad_scale: 0.1 recency_boost: True
=== KELLY ===
KELLY_RET_STD from y_train tail: 0.011000845567985884


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


=== FIT RESULTS ===
XGB best_iteration: 22
XGB2 best_iteration: 57
LGB best_iteration_: 582
=== TEST SANITY ===
test_date_ids: 10 test_features_raw: 10
=== STREAM BUFFER SEED ===
STREAM_COLS: 95
HIST_SEED_N: 400 HIST_MAX_ROWS: 1200

=== PRED CALL 1 ===
incoming test shape: (1, 99)
incoming date_id: [8980] 
cols: 99
hist shape: (401, 95)
hist tail date_id: [8976, 8977, 8978, 8979, 8980]
engineered test_base shape: (1, 261)
top null-frac cols (first 10):
D1              0.0
P2              0.0
P2_div_E17      0.0
P2_roll_z_10    0.0
P2_sq           0.0
P3              0.0
P4              0.0
P4_lag_5        0.0
P5              0.0
P5_lag_1        0.0
dtype: float64
xgb: post-fill NaN frac = 0.0
xgb2: post-fill NaN frac = 0.0
extra: post-fill NaN frac = 0.0
lgb: post-fill NaN frac = 0.0
raw pred stats: xgb (-0.00010899108747253194, -0.00010899108747253194) xgb2 (-0.00045943280565552413, -0.00045943280565552413) extra (-0.0011093360919186349, -0.0011093360919186349) lgb (0.0002005101100442